# Modulo 5: Data Wrangling & Parsing PDF con Pandas
---
## Obiettivi
Questo modulo affronterà uno dei problemi più comuni nel Data Analytics aziendale: **l'estrazione e l'armonizzazione di dati non strutturati provenienti da sorgenti disomogenee (PDF)**.

Abilità e competenze sviluppate:
1. Estrarre testo e tabelle grezze da file PDF usando `pdfplumber` e gestire le relative eccezioni.
2. Padroneggiare il **Data Wrangling** con Pandas e NumPy (espressioni regolari, vettorizzazione, gestione valori mancanti `NaN`, cast dei tipi).
3. Progettare una **Pipeline di Normalizzazione** per riallineare dati provenienti da layout totalmente differenti (Laboratori A, B e C).
4. Unificare i dati puliti (`pd.concat`), effettuare controlli di qualità sulle anomalie ed esportare il dataset finale per la dashboard Streamlit.

Prima di iniziare, é necessario installare le librerie necessarie.

In [ ]:
# [SETUP] Installazione delle librerie necessarie nell'ambiente Google Colab
%pip install pdfplumber reportlab pandas numpy

#test
import pdfplumber
import pandas as pd
import numpy as np
import re
import os

print("Setup completato! Versione Pandas:", pd.__version__)

Con il seguente script si genereranno 3 file PDF di esempio da utilizzare come test per gli script successivi.

In [ ]:
# [SETUP DATI DEMO] Generazione automatica dei 3 PDF di test (Laboratorio A, B e C)
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

def genera_pdf_demo():
    styles = getSampleStyleSheet()

    # 1. Laboratorio A (Layout Tabellare Standard)
    doc_a = SimpleDocTemplate("referto_lab_a.pdf", pagesize=letter)
    elements_a = [
        Paragraph("<b>LABORATORIO ALFA - REFERTO ANALISI</b>", styles['Title']),
        Paragraph("Data Prelievo: 2026-03-10 | Punto Prelievo: PNT-01", styles['Normal']),
        Spacer(1, 10),
        Table([
            ['Parametro', 'Risultato', 'Unita', 'Limite_Legge'],
            ['pH', '7.4', 'pH', '6.5-8.5'],
            ['Nitrati', '12.5', 'mg/L', '50.0'],
            ['Piombo', 'N.D.', 'ug/L', '10.0'],
            ['Cloruri', '24.1', 'mg/L', '250.0']
        ], style=[('GRID', (0,0), (-1,-1), 1, colors.black), ('BACKGROUND', (0,0), (-1,0), colors.lightgrey)])
    ]
    doc_a.build(elements_a)

    # 2. Laboratorio B (Nomi colonne diversi + Valore e Unità fusi)
    doc_b = SimpleDocTemplate("referto_lab_b.pdf", pagesize=letter)
    elements_b = [
        Paragraph("<b>BETA ANALYTICS - RAPPORTO DI PROVA</b>", styles['Title']),
        Paragraph("Data: 11/03/2026 | Codice Punto: PNT-02", styles['Normal']),
        Spacer(1, 10),
        Table([
            ['Analisi_Effettuata', 'Esito_Misura', 'Soglia_Max'],
            ['pH', '8.1 pH', '8.5'],
            ['Nitrati', '58.2 mg/L', '50.0'],
            ['Piombo', '2.1 ug/L', '10.0'],
            ['Arsenico', '< LOD', '10.0']
        ], style=[('GRID', (0,0), (-1,-1), 1, colors.black), ('BACKGROUND', (0,0), (-1,0), colors.grey)])
    ]
    doc_b.build(elements_b)

    # 3. Laboratorio C (Layout Trasposto / Chiave-Valore)
    doc_c = SimpleDocTemplate("referto_lab_c.pdf", pagesize=letter)
    elements_c = [
        Paragraph("<b>GAMMA LAB - SCHEDA RISULTATI</b>", styles['Title']),
        Paragraph("Data: 2026/03/12 | Stazione: PNT-01", styles['Normal']),
        Spacer(1, 10),
        Table([
            ['Proprieta', 'Valore'],
            ['pH', '7.1'],
            ['Nitrati', '8.4 mg/L'],
            ['Piombo', '14.2 ug/L'],
            ['Solfati', '110.0 mg/L']
        ], style=[('GRID', (0,0), (-1,-1), 1, colors.black)])
    ]
    doc_c.build(elements_c)

genera_pdf_demo()
print("PDF demo generati con successo: referto_lab_a.pdf, referto_lab_b.pdf, referto_lab_c.pdf")

---
## 1: Parsing PDF con `pdfplumber` e Gestione Eccezioni

* **Perché i PDF sono complessi?** A differenza dei file CSV o dati JSON, il formato PDF è stato progettato per la **visualizzazione/stampa**, non per lo scambio di dati. Un file PDF descrive la posizione grafica di linee e caratteri su un foglio, non la struttura logica di una tabella.
* **Cos'è `pdfplumber`?** È una libreria Python che analizza il layout dei PDF, identificando righe e colonne per estrarre il testo strutturato.
* **Struttura restituita:** La funzione `extract_tables()` restituisce una **lista di liste di stringhe** (`list[list[str]]`). Ogni elemento della lista principale rappresenta una riga, e ogni sotto-lista contiene le celle di quella riga.
* **Gestione Robustezza (Error Handling):** Operare su file esterni comporta rischi (file mancanti, corrotti o con percorsi errati). Per evitare che lo script si interrompa bruscamente si utilizzsno i blocchi `try...except FileNotFoundError` .

In [ ]:
# Estrazione Dati Grezzi da referto_lab_a.pdf
file_path = "referto_lab_a.pdf"

try:
    with pdfplumber.open(file_path) as pdf:
        page = pdf.pages[0]
        testo_grezzo = page.extract_text()
        print("--- TESTO LIBERO ESTRATTO ---")
        print(testo_grezzo)

        tabelle = page.extract_tables()
        print("\n--- TABELLA GREZZA (Lista di liste) ---")
        print(tabelle[0])

except FileNotFoundError:
    print(f"Errore: Il file '{file_path}' non è stato trovato. Verifica il percorso.")
except Exception as e:
    print(f"Si è verificato un errore inatteso: {e}")

## 2. Introduzione a Pandas: Series e DataFrames
Pandas è la libreria di riferimento per la manipolazione dei dati tabellari in Python. Le sue due strutture dati principali sono:
- **Series**: Un array monodimensionale (una singola colonna).
- **DataFrame**: Una struttura bidimensionale (come un foglio Excel), composta da più Series.

In [ ]:
import pandas as pd

# Creazione di un DataFrame da un dizionario
dati_acqua = {
    'Punto_Prelievo': ['PNT-01', 'PNT-01', 'PNT-02'],
    'Data': ['2026-07-28', '2026-07-28', '2026-07-29'],
    'Parametro': ['Nitrati', 'pH', 'Nitrati'],
    'Valore': [45.5, 7.2, 55.0],
    'Unita': ['mg/L', 'pH', 'mg/L']
}

df = pd.DataFrame(dati_acqua)
display(df) # In Colab e Jupyter, display() formatta la tabella in modo elegante

# Esplorazione base
print("\nInformazioni sul DataFrame:")
df.info()

### 💡 Micro-Challenge 1
**Obiettivo:** Convertire la lista di liste estratta `tabelle[0]` dal file referto_lab_a.pdf in un DataFrame Pandas standard, impostando la prima riga come intestazione delle colonne e rimuovendola dai dati.

*Suggerimento:* Usa `pd.DataFrame(dati[1:], columns=dati[0])`.

In [ ]:
# === FAI QUI IL TUO ESPERIMENTO (Micro-Challenge 1) ===
dati_grezzi = tabelle[0]

# 1. Crea il DataFrame con la prima riga come header
df_lab_a_test = pd.DataFrame(dati_grezzi[1:], columns=dati_grezzi[0])

# 2. Visualizza le prime righe e le info sul tipo di dato
print(df_lab_a_test)
print("\nInformazioni sul DataFrame:")
df_lab_a_test.info()

---
## 3: Data Wrangling: Conversione e Pulizia del Dataset (Pandas & NumPy)

1. **Ispezione Iniziale (`.info()` e `.describe()`):** Quando carichiamo dati esterni, Pandas assegna quasi sempre il tipo `object` (stringa) a tutte le colonne. Prima di fare qualsiasi operazione matematica, dobbiamo verificare i tipi con `.info()`.
2. **Vettorizzazione e Moduli `.str`:** Invece di usare cicli `for` per pulire le stringhe riga per riga, Pandas applica le operazioni su un'intera colonna simultaneamente (vettorizzazione). Es: `df['colonna'].str.strip()`.
3. **Regex per l'Estrazione (`.str.extract()`):** Permette di dividere il contenuto di una colonna in due usando gruppi di cattura. Ad esempio, per separare `"12.5 mg/L"` in `12.5` e `mg/L` usiamo l'espressione regolare `r'([\d\.,]+)\s*(.*)'`.
4. **Gestione del Valore Mancante (`np.nan`):** Nella terminologia Data Science, i valori non rilevati, mancanti o errati (es. `"N.D."` o `"< LOD"`) devono essere convertiti nello standard NumPy `np.nan` (Not a Number). Questo consente di effettuare calcoli statistici senza generare errori.

In [ ]:
# Dimostrazione Pratica: Pulizia Avanzata con Regex e Conversione Tipi
df_demo = pd.DataFrame({
    'Parametro': ['pH', 'Nitrati', 'Piombo'],
    'Esito': ['7.4 pH', '58.2 mg/L', '< LOD'],
    'Data': ['2026-03-10', '11/03/2026', '2026/03/12']
})

print("--- DATI ORIGINALI ---")
print(df_demo)

# 1. Separazione di Valore Numerico e Unità di Misura con Regex
df_demo[['Valore_Num', 'Unita']] = df_demo['Esito'].str.extract(r'([\d\.,]+)\s*(.*)')

# 2. Conversione a numero e gestione dei valori non rilevati (< LOD diventa NaN)
df_demo['Valore_Num'] = pd.to_numeric(df_demo['Valore_Num'].str.replace(',', '.'), errors='coerce')

# 3. Conversione intelligente delle date eterogenee
df_demo['Data_Clean'] = pd.to_datetime(df_demo['Data'], format='mixed')

print("\n--- DATI PULITI ---")
print(df_demo[['Parametro', 'Valore_Num', 'Unita', 'Data_Clean']])
print("\nTipi di dato risultanti:")
print(df_demo[['Valore_Num', 'Data_Clean']].dtypes)

### 💡 Micro-Challenge 2
**Obiettivo:** Trasforma la colonna `Soglia` convertendo i valori in formato numerico decimal (`float`). Assicurati che il valore `"N.A."` sia convertito automaticamente in `NaN` senza bloccare l'esecuzione.

*Suggerimento:* Utilizza `pd.to_numeric()` specificando l'argomento `errors='coerce'`.

In [ ]:
# === FAI QUI IL TUO ESPERIMENTO (Micro-Challenge 2) ===
df_soglie = pd.DataFrame({'Soglia': ['50.0', '10.0', 'N.A.', '250.5']})

df_soglie['Soglia_Num'] = pd.to_numeric(df_soglie['Soglia'], errors='coerce')

print(df_soglie)
print("Tipo della colonna:", df_soglie['Soglia_Num'].dtype)

---
## 4: Riallineamento e Normalizzazione di Layout Eterogenei

### Il Problema Aziendale Reale
In un contesto reale, ciascun laboratorio invia i propri referti PDF utilizzando **strutture di tabella completamente diverse**:
* **Laboratorio A:** Nomi colonne standard (`Parametro`, `Risultato`, `Unita`, `Limite_Legge`).
* **Laboratorio B:** Nomi differenti e fusione di valore/unità (`Analisi_Effettuata`, `Esito_Misura` es. *"58.2 mg/L"*).
* **Laboratorio C:** Struttura verticale/trasposta a due colonne (`Proprieta`, `Valore`).

### L'Architettura della Pipeline di Normalizzazione
Per poter unire questi dati in un unico dataset analitico, dobbiamo definire uno **Schema Unificato (Standard)** e creare una funzione di trasformazione specifica per ogni sorgente.

In [ ]:
SCHEMA_STANDARD = ['data_prelievo', 'punto_prelievo', 'laboratorio', 'parametro', 'valore', 'unita', 'limite']

def normalizza_lab_a(file_pdf):
    with pdfplumber.open(file_pdf) as pdf:
        text = pdf.pages[0].extract_text()
        tables = pdf.pages[0].extract_tables()

        data_p = re.search(r'Data Prelievo:\s*([\d-]+)', text).group(1)
        punto_p = re.search(r'Punto Prelievo:\s*([\w-]+)', text).group(1)

        df = pd.DataFrame(tables[0][1:], columns=tables[0][0])
        df = df.rename(columns={
            'Parametro': 'parametro',
            'Risultato': 'valore',
            'Unita': 'unita',
            'Limite_Legge': 'limite'
        })

        df['data_prelievo'] = pd.to_datetime(data_p)
        df['punto_prelievo'] = punto_p
        df['laboratorio'] = 'Alfa'
        df['valore'] = pd.to_numeric(df['valore'].str.replace(',', '.'), errors='coerce')
        df['limite'] = pd.to_numeric(df['limite'].str.replace(',', '.'), errors='coerce')

        return df[SCHEMA_STANDARD]

def normalizza_lab_b(file_pdf):
    with pdfplumber.open(file_pdf) as pdf:
        text = pdf.pages[0].extract_text()
        tables = pdf.pages[0].extract_tables()

        data_p = re.search(r'Data:\s*([\d/]+)', text).group(1)
        punto_p = re.search(r'Codice Punto:\s*([\w-]+)', text).group(1)

        df = pd.DataFrame(tables[0][1:], columns=tables[0][0])
        df = df.rename(columns={'Analisi_Effettuata': 'parametro', 'Soglia_Max': 'limite'})

        df[['valore', 'unita']] = df['Esito_Misura'].str.extract(r'([\d\.,]+)\s*(.*)')

        df['data_prelievo'] = pd.to_datetime(data_p, format='%d/%m/%Y')
        df['punto_prelievo'] = punto_p
        df['laboratorio'] = 'Beta'
        df['valore'] = pd.to_numeric(df['valore'].str.replace(',', '.'), errors='coerce')
        df['limite'] = pd.to_numeric(df['limite'].str.replace(',', '.'), errors='coerce')

        return df[SCHEMA_STANDARD]

print("Funzioni per Lab A e Lab B pronte!")

### 💡 Micro-Challenge 3
**Obiettivo:** Completa la funzione `normalizza_lab_c()` per processare il referto del Laboratorio C (`referto_lab_c.pdf`).

**Specifiche del Laboratorio:**
1. La data nel testo è nel formato `YYYY/MM/DD`.
2. La colonna `Valore` contiene sia numeri che unità (es. *"8.4 mg/L"*).
3. Il limite di legge nel Lab C non è indicato nella tabella, imposta la colonna `limite` a `np.nan`.

In [ ]:
# === FAI QUI IL TUO ESPERIMENTO (Micro-Challenge 3) ===
def normalizza_lab_c(file_pdf):
    with pdfplumber.open(file_pdf) as pdf:
        text = pdf.pages[0].extract_text()
        tables = pdf.pages[0].extract_tables()

        data_p = re.search(r'Data:\s*([\d/]+)', text).group(1)
        punto_p = re.search(r'Stazione:\s*([\w-]+)', text).group(1)

        df = pd.DataFrame(tables[0][1:], columns=tables[0][0])
        df = df.rename(columns={'Proprieta': 'parametro'})

        df[['valore', 'unita']] = df['Valore'].str.extract(r'([\d\.,]+)\s*(.*)')

        df['data_prelievo'] = pd.to_datetime(data_p, format='%Y/%m/%d')
        df['punto_prelievo'] = punto_p
        df['laboratorio'] = 'Gamma'
        df['valore'] = pd.to_numeric(df['valore'].str.replace(',', '.'), errors='coerce')
        df['limite'] = np.nan

        return df[SCHEMA_STANDARD]

df_c_clean = normalizza_lab_c("referto_lab_c.pdf")
print(df_c_clean)

---
## 5: Concatenazione, Quality Check ed Esportazione

1. **Unificazione Dataset (`pd.concat`):** Permette di combinare verticalmente più DataFrames aventi la stessa struttura di colonne (`ignore_index=True`).
2. **Quality Check (Filtering & Anomalie):** Utilizzando maschere booleane, possiamo isolare le righe in cui il valore misurato supera la soglia di legge (`df['valore'] > df['limite']`).
3. **Persistenza Dati (`to_csv`):** Salva il DataFrame pulito in un file CSV destinato alla Dashboard Streamlit (Modulo 6).

In [ ]:
df_a = normalizza_lab_a("referto_lab_a.pdf")
df_b = normalizza_lab_b("referto_lab_b.pdf")
df_c = normalizza_lab_c("referto_lab_c.pdf")

master_df = pd.concat([df_a, df_b, df_c], ignore_index=True)

print("=== MASTER DATASET UNIFICATO ===")
print(master_df)

anomalie = master_df[master_df['valore'] > master_df['limite']]

print("\n=== ANOMALIE RILEVATE (SUPERAMENTO LIMITI DI LEGGE) ===")
if not anomalie.empty:
    print(anomalie[['data_prelievo', 'punto_prelievo', 'laboratorio', 'parametro', 'valore', 'limite']])
else:
    print("Nessuna anomalia rilevata.")

output_path = "referti_acque_clean.csv"
master_df.to_csv(output_path, index=False)
print(f"\nDataset salvato con successo in '{output_path}'!")

---
## Esercitazione Pratica

### Task:
Scrivi uno script Python modulare ed esaustivo che legga automaticamente **tutti i file PDF** presenti in una cartella, identifichi la tipologia di laboratorio in base al testo contenuto, applichi la corretta funzione di normalizzazione ed esporti un unico file CSV riassuntivo.

<details>
<summary><b>Clicca qui per vedere la soluzione</b></summary>

```python
# === SOLUZIONE ESERCITAZIONE ===
import glob

def pipeline_batch_pdf(directory_path="."):
    lista_dataframe = []
    pdf_files = glob.glob(os.path.join(directory_path, "referto_lab_*.pdf"))
    
    print(f"Trovati {len(pdf_files)} file PDF da elaborare...")
    
    for file_path in pdf_files:
        try:
            with pdfplumber.open(file_path) as pdf:
                testo = pdf.pages[0].extract_text()
                
                if "ALFA" in testo:
                    df_temp = normalizza_lab_a(file_path)
                elif "BETA" in testo:
                    df_temp = normalizza_lab_b(file_path)
                elif "GAMMA" in testo:
                    df_temp = normalizza_lab_c(file_path)
                else:
                    print(f"[WARNING] Layout sconosciuto per: {file_path}")
                    continue
                    
                lista_dataframe.append(df_temp)
                print(f"[OK] Elaborato: {file_path}")
                
        except Exception as e:
            print(f"[ERROR] Impossibile elaborare {file_path}: {e}")
            
    if lista_dataframe:
        final_df = pd.concat(lista_dataframe, ignore_index=True)
        final_df.to_csv("dataset_completo_esercitazione.csv", index=False)
        print("\n=== ELABORAZIONE BATCH COMPLETATA ===")
        print(f"Totale record estratti: {len(final_df)}")
        print("\nPercentuale di NaN per colonna:")
        print(final_df.isna().mean() * 100)
        return final_df
    else:
        print("Nessun dato estratto.")
        return None

df_risultato_finale = pipeline_batch_pdf()